# LeukBench on Colab
Runs Paper 1's pipeline on a free Colab GPU. Runtime → Change runtime type → **T4 GPU**.

Upload `leukbench.zip` (from the chat) to the Colab file panel first, or put it in Google Drive and adjust the path.

In [ ]:
!unzip -q -o leukbench.zip -d /content && cd /content/leukbench && pip install -q -r requirements.txt
!cd /content/leukbench && python -I tests/smoke_test.py /tmp | tail -1

## 1. Data
APL-AML ships inside the DeepAPL repository. For C-NMC, AML-Hehr, LMU and BMC, download from TCIA with the NBIA Data Retriever and copy them to Drive (`/content/drive/MyDrive/leukemia_data/...`).

In [ ]:
%cd /content
!git clone -q --depth 1 https://github.com/sidhomj/DeepAPL
# from google.colab import drive; drive.mount('/content/drive')

## 2. DinoBloom weights (Zenodo record 10908163)

In [ ]:
!mkdir -p /content/weights && wget -q -O /content/weights/DinoBloom-B.pth 'https://zenodo.org/records/10908163/files/DinoBloom-B.pth?download=1' && ls -la /content/weights

## 3. Manifest and features

In [ ]:
%cd /content/leukbench
!python -I scripts/build_manifest.py apl_aml --root /content/DeepAPL/Data --out manifests/apl_aml.csv
!python -I scripts/extract_handcrafted.py --manifest manifests/apl_aml.csv --out features/apl_aml_hand.parquet --workers 4
!python -I scripts/extract_embeddings.py --manifest manifests/apl_aml.csv --encoder effnet_b0_imagenet --out features/apl_aml_effnet.npy
!python -I scripts/extract_embeddings.py --manifest manifests/apl_aml.csv --encoder dinobloom_b --weights /content/weights/DinoBloom-B.pth --out features/apl_aml_dino.npy

## 4. E1/E2 — leakage audit and representation benchmark

In [ ]:
!python -I scripts/run_cell_benchmark.py --manifest manifests/apl_aml.csv --task apl_cell_diagnosis \
  --features hand=features/apl_aml_hand.parquet effnet=features/apl_aml_effnet.npy dino=features/apl_aml_dino.npy \
  --models M1=hand:lgbm M2=effnet:lr M3=dino:lr M4=dino+hand:lr \
  --splits image grouped --seeds 0 1 2 --cv-where cohort=Discovery --test-where cohort=Validation \
  --out results/e1_apl_cell_diagnosis

## 5. E4 — patient-level APL vs AML (Discovery → Validation)

In [ ]:
!python -I scripts/run_patient_benchmark.py --manifest manifests/apl_aml.csv --task apl_patient \
  --features hand=features/apl_aml_hand.parquet effnet=features/apl_aml_effnet.npy dino=features/apl_aml_dino.npy \
  --models P1=hand:pool_lr P2=effnet:pool_lr P3=dino:pool_lr P4=dino+hand:pool_lr P5=dino:abmil P6=dino+hand:abmil \
  --cv-where cohort=Discovery --test-where cohort=Validation --seeds 0 1 2 --out results/e4_apl_patient

## 6. Tables

In [ ]:
!python -I scripts/make_tables.py results/e1_apl_cell_diagnosis results/e4_apl_patient --out results/tables.md
from IPython.display import Markdown; Markdown(open('results/tables.md').read())